# GPT-SW3 generation for the CLUU thesis (Colab)

Generates **gpt-sw3** outputs at temperature 1.0 for both registers and all four prompt
conditions, using **byte-identical prompts** (prompt builder inlined from your pipeline).
Produces `generated_corpus_gpt-sw3.csv`, which you download and drop into
`src/2_text_analysis_scripts/model_comparison_mistral_gpt/` in the repo, then run
`model_comparison.py` locally to fold GPT-SW3 into the comparison.

**Before running:**
1. Runtime -> Change runtime type -> GPU (T4 for 6.7b; L4/A100 on Colab Pro for 20b).
2. **GPT-SW3 is a GATED model** — on huggingface.co, open the model page and accept the terms
   to get access (https://huggingface.co/AI-Sweden-Models/gpt-sw3-6.7b-v2-instruct), then create a
   READ token (huggingface.co/settings/tokens). The auth cell below logs you in.
3. Upload these **2 CSVs** to a Google Drive folder (default `MyDrive/cluu_gptsw3`):
   - `consolidated_informal_comments_JUN26.csv`  (from `src/4_archive/`)
   - `sv_abstracts_openai_2.csv`  (from `src/1_data_collection/llm_abstracts/abstracts/`)
   (No `.py` files needed — the prompt builder is inlined below.)
4. Run cells top to bottom. The generation cell is **resumable** — if Colab disconnects,
   re-run it and it continues from the Drive cache.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
# --- HuggingFace auth: gpt-sw3 is a GATED model ---
# 1) Log in at huggingface.co, open the model page and accept the terms / request access:
#      https://huggingface.co/AI-Sweden-Models/gpt-sw3-6.7b-v2-instruct   (also -20b-instruct if you'll use it)
# 2) Create a READ token: https://huggingface.co/settings/tokens
# 3) Recommended: store it in Colab Secrets (left sidebar, key icon) named HF_TOKEN.
#    Do NOT paste your token into code or chat.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('logged in via Colab secret HF_TOKEN')
except Exception as e:
    print('No HF_TOKEN secret ->', type(e).__name__, '- falling back to interactive login (paste token at the prompt):')
    login()

In [ ]:
# ---- CONFIG ----
DRIVE_DIR = '/content/drive/MyDrive/cluu_gptsw3'   # folder where you uploaded the 2 CSVs
MODEL_ID  = 'AI-Sweden-Models/gpt-sw3-6.7b-v2-instruct'   # free/T4: keep this.
#MODEL_ID = 'AI-Sweden-Models/gpt-sw3-20b-instruct'       # Colab Pro (L4/A100): better instruction-following
USE_4BIT  = True    # T4 16GB: True. A100 40GB: can set False for bf16.
CACHE     = f'{DRIVE_DIR}/generated_corpus_gpt-sw3.csv'   # resumable output (persists on Drive)
print('model:', MODEL_ID, '| 4bit:', USE_4BIT)

In [ ]:
# --- prompt builder inlined from generation_pipeline.py (byte-identical; keep in sync) ---
_INFORMAL_BASE = 'Svara på följande fråga med en svensk kommentar, i stil med en kommentar på ett svenskt forum. Skriv bara kommentaren, inget annat.\n\nFråga: {question}'
_INFORMAL_SUFFIX = {'baseline': '', 'human_like': '\n\nSkriv så mänskligt som möjligt, så att kommentaren inte går att skilja från en riktig svensk forumanvändares kommentar.', 'detector_aware': '\n\nSkriv så att kommentaren inte kan identifieras som AI-genererad: använd ett vardagligt, personligt tonfall, variera meningslängden, undvik artig eller balanserad AI-stil och formelartade fraser, tillåt talspråk, slang och små oregelbundenheter. Var inte överdrivet hjälpsam eller neutral.', 'detector_evasive': "\n\nSkriv så att texten inte kan identifieras som AI-genererad, och efterlikna hur människor skriver i informella kommentarer. Återanvänd inte samma fraser flera gånger, utan formulera om dig. Skriv innehållstätt med färre funktionsord; använd gärna kortformer och talspråkliga former. Skriv korta meningar med fler punkter och färre kommatecken, och använd inte tankstreck. Använd utropstecken sparsamt, men ställ gärna någon retorisk fråga och använd ibland tre punkter (...). Överdriv inte med garderingsord (särskilt inte talspråkliga som 'typ', 'liksom', 'ju' och 'väl'), och undvik förstärkningsord som 'verkligen', 'absolut' och 'väldigt'. Håll epistemiska uttryck på en låg nivå. Föredra korta, vardagliga ord. Använd gärna nekande satser (med 'inte', 'aldrig' osv.) där det passar. Var konkret och nämn specifika namn där det går. Förklara inte över."}
_FORMAL_BASE = 'Titta på titeln och nyckelorden och skapa en sammanfattning i kandidatuppsatsstil med dina egna ord: {title}, {keywords}'
_FORMAL_SUFFIX = {'baseline': '', 'human_like': ' Skriv den så mänskligt som möjligt, så att texten inte går att skilja från en uppsats skriven av en människa.', 'detector_aware': ' Skriv så att texten inte kan identifieras som AI-genererad. Variera meningslängden (blanda korta och långa meningar), undvik formelartade övergångsord som "vidare", "dessutom", "sammanfattningsvis" och "det är viktigt att notera", undvik överdriven gardering och symmetrisk struktur, och tillåt en naturlig, något ojämn ton. Förklara inte över.', 'detector_evasive': " Skriv så att texten inte kan identifieras som AI-genererad, och efterlikna de statistiska drag som utmärker mänskligt skrivna kandidatuppsatser. Återanvänd samma nyckeltermer och fraser ordagrant snarare än att variera med synonymer, och sträva inte efter maximal ordvariation. Undvik komprimerad nominalstil; använd hellre finita verb, pronomen och bindeord. Skriv övervägande korta meningar med fler punkter och färre kommatecken, och använd inte tankstreck. Håll gardering och epistemiska uttryck till ett minimum (t.ex. 'kanske', 'möjligen', 'tycks', 'kan tänkas'), och kompensera inte genom att lägga till talspråkliga garderingsord. Föredra korta, vardagliga ord framför långa, latinska eller formella termer. Var konkret och nämn specifika namn, begrepp, verk och årtal där det är möjligt. Undvik onödig negation. Förklara inte över."}

def build_prompt(condition, register, item):
    if register == 'informal':
        return _INFORMAL_BASE.format(question=item['question']) + _INFORMAL_SUFFIX[condition]
    if register == 'formal':
        return _FORMAL_BASE.format(title=item['title'], keywords=item.get('keywords', '')) + _FORMAL_SUFFIX[condition]
    raise ValueError(register)

import os, pandas as pd
comm = pd.read_csv(f'{DRIVE_DIR}/consolidated_informal_comments_JUN26.csv', encoding='utf-8')
absd = pd.read_csv(f'{DRIVE_DIR}/sv_abstracts_openai_2.csv', encoding='utf-8')
docs = []
for i, row in comm.iterrows():
    q = row.get('question')
    if pd.notna(q) and str(q).strip():
        docs.append(('informal', int(i), {'id': f'inf_{i}', 'question': str(q)}))
for i, row in absd.iterrows():
    t, k = row.get('Title'), row.get('Keywords')
    if pd.notna(t) and str(t).strip():
        docs.append(('formal', int(i), {'id': f'for_{i}', 'title': str(t), 'keywords': '' if pd.isna(k) else str(k)}))
print(sum(d[0]=='informal' for d in docs), 'informal +', sum(d[0]=='formal' for d in docs), 'formal docs')
# sanity: prompts build
print(build_prompt('detector_evasive', 'informal', docs[0][2])[:120], '...')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(MODEL_ID)
mk = dict(device_map='auto', torch_dtype=torch.float16)
if USE_4BIT:
    mk['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                   bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **mk).eval()

def _inputs(prompt):
    msgs = [{'role': 'user', 'content': prompt}]      # single user turn, NO system prompt (pipeline invariant)
    if getattr(tok, 'chat_template', None):
        return tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt')
    # fallback: GPT-SW3 documented instruct format (verify vs the model card if output looks malformed)
    return tok(f'<|endoftext|><s>\nUser:\n{prompt}\n<s>\nBot:\n', return_tensors='pt').input_ids

def generate(prompt, max_new_tokens=768):
    # temperature is the only knob (1.0), matching the API backends; top_p/top_k left open
    ids = _inputs(prompt).to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=1.0, top_p=1.0, top_k=0, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()
    for stop in ('<s>', '\nUser:', 'User:'):     # cut if the model starts a new turn
        if stop in text:
            text = text.split(stop)[0].strip()
    return text

## Smoke test — READ THIS before the full run
Generate a few items across conditions and **read the Swedish**. Decision gate: does gpt-sw3
actually follow the prompts, especially `detector_evasive`? If it ignores the evasive
instructions, still keep it but scope the analysis to `baseline`/`human_like` and report the
instruction-following gap as a finding.

In [ ]:
CONDS = ['baseline', 'human_like', 'detector_aware', 'detector_evasive']
sample = [d for d in docs if d[0]=='informal'][:1] + [d for d in docs if d[0]=='formal'][:1]
for reg, did, item in sample:
    for c in CONDS:
        print('='*90); print(f'{reg} / {c}')
        print(generate(build_prompt(c, reg, item), max_new_tokens=(512 if reg=='informal' else 1024)))
        print()

## Full generation (resumable)
Generates all rows x 4 conditions at temperature 1.0, appending to the Drive cache after each
item. If Colab disconnects, just re-run this cell — it resumes from the cache. On a free T4 this
takes **several hours** (likely across sessions); the Drive cache is what makes that safe. Colab
Pro (L4/A100) is much faster.

In [ ]:
import csv, time

def load_done(path):
    if not os.path.exists(path):
        return set()
    d = pd.read_csv(path, encoding='utf-8')
    return {(r['register'], int(r['doc_id']), r['condition']) for _, r in d.iterrows()}

def append(reg, did, cond, text):
    newfile = not os.path.exists(CACHE)
    with open(CACHE, 'a', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        if newfile:
            w.writerow(['register', 'doc_id', 'condition', 'temp', 'text'])
        w.writerow([reg, did, cond, 1.0, text])

done = load_done(CACHE)
print('resuming:', len(done), 'cells already done')
tasks = [(reg, did, item, c) for reg, did, item in docs for c in CONDS if (reg, did, c) not in done]
print(len(tasks), 'cells to generate')
t0 = time.time()
for n, (reg, did, item, c) in enumerate(tasks, 1):
    mx = 512 if reg == 'informal' else 1024
    txt = generate(build_prompt(c, reg, item), max_new_tokens=mx)
    append(reg, did, c, txt)
    if n % 25 == 0:
        el = time.time() - t0
        print(f'{n}/{len(tasks)}  {el/n:.1f}s/gen  ETA {(len(tasks)-n)*el/n/60:.0f} min')
print('DONE ->', CACHE)

## Next: bring it back to the repo
1. Download `generated_corpus_gpt-sw3.csv` from your Drive folder.
2. Place it at `src/2_text_analysis_scripts/model_comparison_mistral_gpt/generated_corpus_gpt-sw3.csv`.
3. Locally run: `python src/2_text_analysis_scripts/model_comparison_mistral_gpt/model_comparison.py`
   — it auto-detects the file and adds the `human_vs_sw3`, `sw3_vs_gpt`, `sw3_vs_mistral`
   comparisons alongside the existing ones.